# Dane niezbalansowane

<div style="text-align: center;"><img src=".//Images//Dogs and cats.png" alt="Hiperparametry" width="400" height="120" style="margin: 10px; "/></div>

Dane niezbalansowane (ang. imbalanced data) to takie zbiory danych, w których liczba obserwacji (przykładów) należących do poszczególnych kategorii (klas) jest wyraźnie nierówna. Przykładem może być sytuacja, w której 90% danych należy do jednej klasy, a tylko 10% do pozostałych. Taka dysproporcja występuje często w problemach typu wykrywanie fraudów, analiza medyczna (rzadkie choroby), czy filtrowanie spamu.

**Dlaczego to istotne?**

* **Zachwianie miary dokładności (accuracy)**: Model może osiągać wysoką trafność, „ignorując” mniejszościową klasę i nadal uzyskiwać pozornie dobry wynik.
* **Trudności w trenowaniu modeli**: Tradycyjne algorytmy uczenia maszynowego nie uwzględniają domyślnie nierównomiernego rozkładu klas, co prowadzi do niedouczenia się wzorców istotnych dla klas mniejszościowych.
* **Właściwy dobór metryk**: Przy danych niezbalansowanych bardziej miarodajne stają się metryki uwzględniające rozkład klasy, np. f1, AUC-ROC, Precision, Recall.

Możemy wyróżnić trzy główne podejścia do radzenia sobie z danych niezbalansowanych:

* **Na poziomie danych** (ang. Data-level methods), które modyfikują dostępne instancje problemu w celu jego zbalansowania. Tutaj najbardziej popularne podejścia to **oversampling**, który generuje sztuczne instancje klasy mniejszościowej oraz **undersampling**, który pozbywa się instancji klasy większościowej.

* **Na poziomie algorytmów** (ang. Algorithm-level methods), które modyfikują istniejące algorytmy uczenia maszynowego, aby zredukować ich bias w kierunku klasy większościowej.

* **Podejścia hybrydowe** (ang. Hybrid methods), łączące oba wyżej opisane rozwiązania.

Zajmiemy się głównie podejściem na poziomie danych, a więc oversamplingiem i undersamplingiem. Wybierzemy w tym celu po dwie metody z każdej grupy:

**Oversampling**
* **Random Oversampling** (ROS) - najprostsza koncepcyjnie metoda oversamplingu, która wyrównuje liczność klas poprzez powielanie istniejących instancji klasy mniejszościowej (poprzez losowanie ze zwracaniem).
* **Synthetic minority over-sampling technique** (SMOTE) - klasyczny algorytm oversamplingu, który generuje syntetyczne instancje klasy mniejszościowej pomiędzy istniejącymi instancjami tej klasy a ich sąsiadami (również z klasy mniejszościowej).
  
**Udersampling**
* **Random Undersampling** (RUS) - adekwatnie do ROS, tylko w drugą stronę - usuwamy losowe instancje klasy większościowej aż nie zbalansujemy problemu. Niestety, możemy przez to stracić przydatne informacje.
* **Condensed Nearest Neighbour** (CNN) - technika undersamplingu, która usuwa te instancje klasy większościowej, które można poprawnie zaklasyfikować korzystając z 1-NN (Algorytmu najbliższych sąsiadów z jednym sąsiadem). Dzięki temu zostawiamy instancje klasy większościowej, które znajdują się blisko granicy decyzyjnej.

# Metryki

## Metryki bazowe

Zacznijmy od macierzy konfuzji:

<div style="text-align: center;"><img src=".//Images//confusion_matrix.png" alt="cn" width="200" height="120" style="margin: 10px; "/></div>

Musimy także pamiętać, że aby poniższe metryki były informatywne, klasa mniejszościowa musi być pozytywna.

**Sensitivity** (**Recall**) mówi nam o tym, jak dobrze nasz algorytm radzi sobie z rozpoznawaniem obiektów klasy mniejszościowej. Dzięki niemu wiemy, ile obiektów uznaliśmy za należące do klasy mniejszościowej w stosunku do tego, ile faktycznie ich jest.

$$Recall = \frac{TP}{TP + FN}$$

**Precision** informuje nas o tym, jak dokładny jest nasz algorytm podczas rozpoznawania klasy mniejszościowej. Dzięki temu wiemy, ile z obiektów, które przyporządkowaliśmy do tej klasy, faktycznie do niej należy.

$$Precision = \frac{TP}{TP + FP}$$

**Specificity** działa jak recall, ale dla klasy większościowej.

$$Specificity = \frac{TN}{TN + FP}$$

## Metryki zagregowane

**f1 score**, jest to średnia harmoniczna precision i recall.

$$f_1 = 2 * \frac{Precision * Recall}{Precision + Recall}$$

**Geometric mean score** (G-mean) zazwyczaj stanowi pierwiastek iloczynu recall i precision.

$$Gmean = \sqrt{Recall * Specificity}$$

**Balanced accuracy** (BAC) - zbalansowana dokładność to po prostu średnia recall i specificity.

$$BAC = \frac{Recall + Specificity}{2}$$

# Przykład: Klasyfikacja pozytywnego wyniku biopsji szyjki macicy

<div style="text-align: center;"><img src=".//Images//Zadanie.jpeg" alt="zadanie" width="400" height="120" style="margin: 10px; "/></div>

**Zbiór danych**: Cervical Cancer (Risk Factors)
[UCI ML Repository – Cervical Cancer](https://archive.ics.uci.edu/dataset/383/cervical+cancer+risk+factors)

- 858 przykładów
- 32 cechy (np. wiek, liczba partnerów seksualnych, palenie, itp.)
- 4 różne etykiety wyjściowe (binarnie oznaczone):
  - `Hinselmann`
  - `Schiller`
  - `Citology`
  - `Biopsy`

**Uwaga**: *W "sieci" można odszukać wiele przykładów osiągania wysokich wartości metryk oceny modeli, jednak zazwyczaj nie usuwano z danych pozostałych cech określających stan choroby :)*

**Cel zadania:** Zbudować model klasyfikacyjny, który przewiduje wynik **`Biopsy`** (czy biopsja wykazała zmiany nowotworowe).


**Nierównowaga klas w `Biopsy`:**
- `Pozytywne`: 55
- `Negatywne`: 803
- Tylko **~6.4%** pozytywnych przypadków.

## Zadanie - balansowanie na poziomie algorytmów

### Macierz kosztów

**Macierz kosztów** (ang. *cost matrix*) to sposób formalnego określenia, **jak bardzo kosztowne są błędy klasyfikacji** — np. pomylenie zdrowej osoby z chorą może być znacznie mniej groźne niż odwrotnie. Zamiast traktować wszystkie błędy jednakowo (jak w accuracy), możesz nadać im **różne wagi**.

---

**Przykład macierzy kosztów (dla problemu binarnego)**:

|                       | Przewidziano klasę 0 | Przewidziano klasę 1 |
|-----------------------|----------------------|----------------------|
| **Prawdziwa klasa 0** | koszt = 0            | koszt = 1            |
| **Prawdziwa klasa 1** | koszt = 10           | koszt = 0            |

To znaczy:
- **False Positive** (zdrowy uznany za chorego) → koszt 1  
- **False Negative** (chory uznany za zdrowego) → koszt 10  
- Trafienia nie kosztują nic (koszt 0)

---

1. **W `sklearn` przez `class_weight`** (prosta wersja)

```python
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight={0: 1, 1: 10})
```

lub automatycznie:

```python
model = LogisticRegression(class_weight='balanced')
```

2. **Ręczne ważenie próbek (`sample_weight`) przy uczeniu lub ocenie**

```python
from sklearn.utils.class_weight import compute_sample_weight

weights = compute_sample_weight(class_weight={0: 1, 1: 10}, y=y_train)
model.fit(X_train, y_train, sample_weight=weights)
```

---

### Rozwiązanie - wybór algorytmu

`pip install imbalanced-learn` lub `pip install imblearn`

In [ ]:
# Szykuje się wersja sklearn 1.7, zatem zaczyna się pojawiać sporo uwag :)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Import bibliotek
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, recall_score, accuracy_score, precision_score, balanced_accuracy_score
from imblearn.metrics import geometric_mean_score

In [ ]:
# Wczytanie danych
df = pd.read_csv("./Data/risk_factors_cervical_cancer.csv")

# Konwersja '?' → NaN i rzutowanie
df = df.replace('?', np.nan).astype(float)

# Wybór cech
X_raw = df.drop(columns=['Hinselmann', 'Schiller', 'Citology', 'Biopsy'])
y = df['Biopsy'].astype(int)

In [ ]:
# Imputacja braków (średnia lub kNN)
# imputer = SimpleImputer(strategy='mean')
# X = pd.DataFrame(imputer.fit_transform(X_raw), columns=X_raw.columns)

imputer = KNNImputer(n_neighbors=5)
X = pd.DataFrame(imputer.fit_transform(X_raw), columns=X_raw.columns)

In [ ]:
# Sprawdź rozkład klas 
print("Rozkład klas:")
print(y.value_counts())

**Model i metryki - balansowanie na podstawie algorytmu**

In [ ]:
lr_model = LogisticRegression(max_iter = 1000, random_state=42, class_weight=None)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight=None)

lr_model_balanced = LogisticRegression(max_iter = 1000, random_state=42, class_weight='balanced')
rf_model_balanced = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

weights = class_weight={0: 1, 1: 10}

lr_model_manual_class_weight = LogisticRegression(max_iter = 1000, random_state=42, class_weight=weights)
rf_model_manual_class_weight = RandomForestClassifier(n_estimators=100, random_state=42, class_weight=weights)

models = {
    "LogisticRegression (no class_weight)": lr_model,
    "RandomForest (no class_weight)": rf_model,
    "LogisticRegression (balanced)": lr_model_balanced,
    "RandomForest (balanced)": rf_model_balanced,
    "LogisticRegression (manual class_weight)": lr_model_manual_class_weight,
    "RandomForest (manual class_weight)": rf_model_manual_class_weight,
}

**class_weight** dict lub 'balanced', domyślnie=None

Wagi powiązane z klasami w danych. Jeśli nie podano, wszystkie klasy powinny mieć wagę jeden.
W trybie „balanced” wartości y są wykorzystywane do automatycznego dostosowywania wag odwrotnie proporcjonalnie do częstości klas w danych wejściowych, jako $n\_samples / (n\_classes * np.bincount(y))$

Należy pamiętać, że wagi te zostaną pomnożone przez **sample_weight** (przekazane przez metodę fit), jeśli **sample_weight** zostanie określony.

In [ ]:
scoring = {
    'f1': make_scorer(f1_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0),
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score, zero_division=0),
    'bac': make_scorer(balanced_accuracy_score),
    'gmean': make_scorer(geometric_mean_score)
}

In [ ]:
# Walidacja - stratyfikowana
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
print("\n==== Porównanie metod ====\n")

for model_name, model in models.items():
    results = cross_validate(model, X, y, cv=skf, scoring=scoring, return_train_score=False)
    
    f1 = np.mean(results['test_f1'])
    recall = np.mean(results['test_recall'])
    accuracy = np.mean(results['test_accuracy'])
    precision = np.mean(results['test_precision'])
    bac = np.mean(results['test_bac'])
    gmean = np.mean(results['test_gmean'])

    print(f"Name: {model_name:<40}  Accuracy: {accuracy:.4f}  f1: {f1:.4f}  Recall: {recall:.4f}  Precision: {precision:.4f}  BAC: {bac:.4f}  G-Mean: {gmean:.4f}")

----

Wyniki wyraźnie pokazują, że **sama miara Accuracy nie wystarcza przy analizie problemów z niezbalansowanymi klasami**. Modele bez wag klas (`no class_weight`) osiągają bardzo wysoką dokładność (~93%), jednak praktycznie nie wykrywają klasy mniejszościowej – w przypadku LogisticRegression recall wynosi 0.0, a w RandomForest tylko 0.0545. Oznacza to, że model uczy się niemal wyłącznie klasy większościowej.

Zastosowanie `class_weight='balanced'` w LogisticRegression znacząco zmieniło jego charakterystykę – dokładność spadła do ~77%, ale recall wzrósł do ~42%, a f1 do ~0.19. Model zaczął realnie rozpoznawać klasę mniejszościową, choć kosztem większej liczby pomyłek w klasie większościowej. Odpowiednie ważenie klas wymusiło większą „czujność” wobec przypadków trudniejszych do wykrycia.

W przypadku RandomForest z tym samym ustawieniem (`balanced`) wagi klas nie przyniosły niemal żadnej poprawy – zarówno recall (0.0364), jak i f1 (0.0583) pozostały na bardzo niskim poziomie, zbliżonym do wersji bez wag. Dokładność (~93%) utrzymała się, ale nie to jest najważniejsze – model nadal niemal całkowicie ignoruje klasę mniejszościową.

Wnioski są jasne: LogisticRegression lepiej wykorzystuje informację o niezbalansowaniu klas przy użyciu `class_weight='balanced'`, znacząco poprawiając detekcję klasy mniejszościowej. W tej konfiguracji RandomForest okazuje się niewrażliwy na to ustawienie i nie dostarcza realnych korzyści w tym zakresie. 

Metryki "balasowane" (BAC, G-mean) lepiej oddają charakter wyniku.

Wydaje się zatem, że najlepszym wyborem z zaprezentowanych wydaje się **LogisticRegression (balanced)** lub **LogisticRegression (manual class_weight)**. Uzyskały one wysokie Recall i zdecydowanie wyższe f1 w porównaniu z pozostałymi modelami, co oznacza, że znacznie skuteczniej wykrywały klasę mniejszościową, nawet kosztem spadku dokładności.

## Zadanie - balasowanie danych

### Oversampling

Przegląd najważniejszych metod oversamplingu dostępnych w bibliotece **`imblearn.over_sampling`**:

---
**RandomOverSampler**
Najprostsza metoda – losowo powiela istniejące instancje klasy mniejszościowej.
* Zaleta: szybka i łatwa.
* Wada: ryzyko przeuczenia (overfitting), bo nie dodaje nowej informacji.

---
**SMOTE** (Synthetic Minority Over-sampling Technique)
Tworzy syntetyczne próbki klasy mniejszościowej przez interpolację pomiędzy istniejącymi punktami.
* Zaleta: bardziej zróżnicowane dane.
* Wada: może „rozciągać” dane w obszary, gdzie klasa mniejszościowa normalnie nie występuje.

---
**Borderline-SMOTE**
Wariant SMOTE, który generuje nowe próbki blisko granicy decyzyjnej, czyli tam, gdzie klasy się mieszają.
* Zaleta: skupia się na trudnych przypadkach.
* Wada: może pogorszyć sytuację, jeśli dane są bardzo zaszumione.

---
**SVMSMOTE**
Korzysta z SVM, aby znaleźć trudne przypadki do wygenerowania syntetycznych przykładów.
* Zaleta: bardziej precyzyjny dobór punktów.
* Wada: wolniejszy, wymaga więcej obliczeń.

---
**ADASYN** (Adaptive Synthetic Sampling)
Podobny do SMOTE, ale generuje więcej syntetycznych próbek tam, gdzie model ma większe trudności z rozpoznaniem klasy mniejszościowej.
* Zaleta: dynamiczne dopasowanie do lokalnej trudności klasyfikacji.
* Wada: może wprowadzać szum w dane.

---

### Undersampling

---

Przegląd najważniejszych metod **undersamplingu** dostępnych w bibliotece **`imblearn.under_sampling`**:

---
**RandomUnderSampler**  
Losowo usuwa instancje klasy większościowej aż do osiągnięcia równowagi między klasami.  
- Zaleta: bardzo szybka, prosta w użyciu, eliminuje nadmiarowe dane.  
- Wada: może przypadkowo usunąć istotne informacje, co może pogorszyć jakość modelu.

---
**TomekLinks**  
Wyszukuje pary próbek (Tomek links), które należą do różnych klas i są dla siebie najbliższymi sąsiadami – usuwa z nich te z klasy większościowej.  
- Zaleta: poprawia „czystość” granicy decyzyjnej, zachowując strukturę danych.  
- Wada: działa tylko przy wyraźnie oddzielonych klasach; usuwa stosunkowo niewiele przykładów.

---
**Edited Nearest Neighbours (ENN)**  
Dla każdej próbki sprawdza jej k najbliższych sąsiadów – jeśli większość sąsiadów wskazuje inną klasę, próbka jest usuwana.  
- Zaleta: eliminuje „hałaśliwe” przykłady, poprawia jakość danych przy granicy klas.  
- Wada: może prowadzić do usunięcia zbyt wielu punktów, jeśli klasy są blisko siebie.

---
**Condensed Nearest Neighbour (CNN)**  
Usuwa wszystkie próbki klasy większościowej, które **nie są konieczne** do poprawnej klasyfikacji danych przy użyciu 1-NN.  
- Zaleta: pozostawia tylko najważniejsze dane przy granicy decyzyjnej.  
- Wada: może silnie zredukować dane – nadaje się głównie do problemów z wyraźną separacją klas.

---
**Neighbourhood Cleaning Rule (NCR)**  
Łączy ENN i inne reguły czyszczenia – usuwa próbki klasy większościowej, jeśli są źle klasyfikowane przez sąsiadów lub zakłócają klasyfikację klasy mniejszościowej.  
- Zaleta: bardziej zaawansowana filtracja nieistotnych punktów.  
- Wada: wolniejsza, wymaga więcej pamięci i może być podatna na szum.

---

## Rozwiązanie - balansowanie danych

In [ ]:
# zdefiniowanie klasyfikatorów, technik preprocessingu i metryk
from imblearn.over_sampling import (
  RandomOverSampler, 
  SMOTE, 
  BorderlineSMOTE, 
  SVMSMOTE, 
  ADASYN
)

from imblearn.under_sampling import (
    RandomUnderSampler,
    TomekLinks,
    EditedNearestNeighbours,
    CondensedNearestNeighbour,
    NeighbourhoodCleaningRule
)

from imblearn.combine import SMOTEENN, SMOTETomek # metody hybrydowe

from imblearn.metrics import geometric_mean_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, precision_score, f1_score, balanced_accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split
from collections import Counter
from sklearn.preprocessing import StandardScaler

In [ ]:
pd.set_option('display.max_columns', None)        # pokazuj wszystkie kolumny
pd.set_option('display.expand_frame_repr', False) # nie łam wierszy na kilka linii

In [ ]:
# Walidacja - stratyfikowana
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Model
lr_model_balanced = LogisticRegressionCV(max_iter = 2000, random_state=42, cv=skf, class_weight='balanced')
model = lr_model_balanced
# Podział
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Skalowanie
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
preprocs = {
    'none': None,
    'ros':   RandomOverSampler(random_state=42),
    'smote': SMOTE(random_state=42),
    'rus':   RandomUnderSampler(random_state=42),
    'cnn':   CondensedNearestNeighbour(random_state=42),
}
metrics = {
    "recall": recall,
    'precision': precision,
    'f1': f1_score,
    'g-mean': geometric_mean_score,
    'bac': balanced_accuracy_score,
}

In [ ]:
# oversampling + undersampling
all_samplers = {
    'Bez balansowania': None,
    'Oversample: RandomOverSampler': RandomOverSampler(random_state=42),
    'Oversample: SMOTE': SMOTE(random_state=42, k_neighbors=3), # mniejsza ilość sąsiadów
    'Oversample: BorderlineSMOTE': BorderlineSMOTE(random_state=42, k_neighbors=3, kind='borderline-2'),
    'Oversample: ADASYN': ADASYN(random_state=42, n_neighbors=3),
    'Undersample: RandomUnderSampler': RandomUnderSampler(random_state=42),
    'Undersample: TomekLinks': TomekLinks(sampling_strategy='auto'),
    'Undersample: EditedNearestNeighbours': EditedNearestNeighbours(sampling_strategy='majority', n_neighbors=3),
    'Hybrid: SMOTEENN': SMOTEENN(random_state=42),
    'Hybrid: SMOTETomek': SMOTETomek(random_state=42)
}

In [ ]:
# Zbieranie wyników
results = []

for name, sampler in all_samplers.items():
    if sampler is None:
        # Bez przetwarzania
        X_res, y_res = X_train.copy(), y_train.copy()
    else:
        X_res, y_res = sampler.fit_resample(X_train, y_train)

    # Trening i predykcja
    model.fit(X_res, y_res)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]  # prawdopodobieństwo klasy 1

    # Liczebność klas po resamplingu
    class_counts = Counter(y_res)
    maj_count = max(class_counts.values())
    min_count = min(class_counts.values())

    # Metryki
    results.append({
        'method': name,
        'majority_count': maj_count,
        'minority_count': min_count,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_score),
        'balanced_accuracy_score': balanced_accuracy_score(y_test, y_pred),
    })

In [ ]:
def highlight_row(s):
    return ['font-weight: bold' if s['method'] == 'Bez balansowania' else '' for _ in s]

# Tabela wyników
results_df = pd.DataFrame(results)
results_df_sorted = results_df.sort_values(by='f1', ascending=False)
results_df_sorted.style.apply(highlight_row, axis=1)

----

Najlepiej wypadła metoda **SMOTEENN**, osiągając najwyższy recall (0.45), najwyższy ROC AUC (0.64) i najwyższą balanced accuracy (0.60), co czyni ją najlepszym wyborem do wykrywania rzadkich przypadków. **SMOTETomek** uzyskał porównywalny f1 i recall (0.27), ale przy wyższej ogólnej accuracy (0.85), więc lepiej sprawdzi się jako kompromis między skutecznością a jakością klasyfikacji ogólnej. Pozostałe metody, szczególnie **ADASYN** i **BorderlineSMOTE**, miały wyraźnie niższe metryki i nie poprawiły wyników względem braku przetwarzania. Undersampling utrzymywał stały poziom, ale nie wnosił znaczącej poprawy.

----

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
# Wybór metryk do wykresu radarowego
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'balanced_accuracy_score']
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]  # zamknięcie okręgu

# Przygotowanie wykresu
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
colors = plt.cm.tab10.colors

for idx, row in results_df.iterrows():
    values = row[metrics].tolist()
    values += values[:1]
    ax.plot(angles, values, label=row['method'], color=colors[idx % len(colors)])
    ax.fill(angles, values, alpha=0.1, color=colors[idx % len(colors)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics)
ax.set_yticklabels([])
ax.set_title("Radarowy wykres porównania metod balansowania", size=14)
ax.legend(loc='upper right', bbox_to_anchor=(1.5, 1.1))

plt.tight_layout()
plt.show()

# Podsumowanie

Oczywiście Nasze dotychczasowe testy metod balansowania (np. RandomUnderSampler, TomekLinks, SMOTEENN itd.) nie dają jednoznacznej odpowiedzi na to, która metoda jest „najlepsza”. Porównywanie metryk takich jak recall czy f1 przed i po balansowaniu może coś zasugerować, ale nie powinno być traktowane jako dowód skuteczności tych technik. W praktyce, samo zastosowanie balansowania to nic innego jak dodanie kolejnego hiperparametru do układanki – i to takiego, który wcale nie musi przynieść poprawy.

Czasem balansowanie (szczególnie undersampling) może nawet pogorszyć sytuację, bo tracimy informacje. Oversampling z kolei zwiększa zbiór danych, co oznacza dłuższy czas trenowania modeli i większy koszt strojenia parametrów, zwłaszcza przy złożonych algorytmach. Dodatkowo, jeżeli chcemy analizować szerzej model, to sztucznie wygenerowane dane mogą zniekształcać jego obraz.

Nie twierdzę, że balansowanie nie ma sensu – wręcz przeciwnie, w niektórych sytuacjach może być bardzo użyteczne. Przykład: wiemy, że w rzeczywistej aplikacji proporcje klas będą zbliżone, ale w danych treningowych mamy duży brak równowagi (np. kosztowne etykietowanie pozytywnej klasy). Wtedy balansowanie może być bardziej zgodne z rzeczywistością niż niebalansowane dane treningowe – zarówno w fazie treningu, jak i końcowej walidacji.

**Ale nie zawsze musimy sięgać po SMOTE, ADASYN, czy inne zaawansowane techniki. W wielu przypadkach wystarczy dobrze przemyślane przypisanie wag klasom lub zdefiniowanie macierzy kosztów. Takie podejście jest prostsze, szybsze, bardziej przejrzyste i często wystarczające – o ile oczywiście model i narzędzia, z których korzystamy, na to pozwalają.**

# Bibliografia 

https://metsi.github.io/2020/05/15/kod8.html

https://miroslawmamczur.pl/niezbalansowane-dane-klasyfikacyjne-na-ratunek-smote/

https://danetyka.com/statystyka-w-data-science/smote-niezbalansowane-dane/

https://baotramduong.medium.com/optimizing-binary-classifiers-tackling-unbalanced-datasets-with-1-minority-class-fe83d801fd09

In [ ]:
import sklearn
print(sklearn.__version__)